# Baseline 4 — Xing & Du (2019): Deep MLP + intervention prioritization
**Reference:** Xing & Du, *J. Educ. Comput. Res.* 57(3):547–570, 2019. 
DOI: 10.1177/0735633118757015---
### What this reproduces
Deep MLP for personalized risk scoring + top-k recall as a proxy forintervention prioritization (the paper's core operational metric).
### V2 setup notes (this re-run)
- Reads from **`dbo.tbl_graphgrued_weekly_active_base_balanced`** (V4 cohort + balanced label).
- Uses **TRUE weekly columns** `*_week1..*_week4` for windowed views (no linear scaling).
- Targets: `y_5class` (A=0..E=4) and `y_binary` (is_completed) — both already in V4.
- Stratified 8:1:1 split (seed=42), median-impute + standardize fit on train only.
### Model recipe- DeepMLP (128-64-32) with dropout 0.3.
- Risk score r = P(D)+P(E) for 5-class, P(not_completed) for binary.- Reports recall@5%, recall@10%, recall@20% on top-ranked test learners.

In [3]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import os, time, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)

MODEL_BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
WINDOWS = [1, 2, 4]
LABEL_ORDER_5 = ["A", "B", "C", "D", "E"]

print("Imports OK — reading from", MODEL_BASE_TABLE)


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 31, Finished, Available, Finished, False)

Imports OK — reading from dbo.tbl_graphgrued_weekly_active_base_ordered5


In [4]:
# ============================================================
# 1) LOAD MODEL BASE TABLE (V4 balanced)
# ============================================================
SAMPLE_FRACTION = 1.0
MAX_ROWS = 300_000

sdf = spark.table(MODEL_BASE_TABLE)
total_rows = sdf.count()
print(f"V4 cohort rows: {total_rows:,}")

if SAMPLE_FRACTION < 1.0:
    sdf = sdf.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

pdf = sdf.toPandas()
if len(pdf) > MAX_ROWS:
    g = pdf.groupby("risk_label_5_idx", group_keys=False)
    pdf = g.apply(lambda d: d.sample(
        n=max(1, int(len(d) * MAX_ROWS / len(pdf))),
        random_state=SEED)).reset_index(drop=True)

print(f"Working dataframe shape: {pdf.shape}")
print("\n5-class distribution after sampling:")
print(pdf["risk_label_5_idx"]
      .map({0:"A",1:"B",2:"C",3:"D",4:"E"})
      .value_counts(normalize=True).round(4)
      .reindex(LABEL_ORDER_5))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 32, Finished, Available, Finished, False)

V4 cohort rows: 100,409
Working dataframe shape: (100409, 71)

5-class distribution after sampling:
risk_label_5_idx
A    0.0222
B    0.0812
C    0.0769
D    0.6411
E    0.1785
Name: proportion, dtype: float64


In [5]:
# ============================================================
# 2) LABELS — already in V4; just alias
# ============================================================
pdf["y_5class"] = pdf["risk_label_5_idx"].astype(int)
pdf["y_binary"] = pdf["is_completed"].fillna(0).astype(int)

print("y_5class distribution:")
counts5 = pdf["y_5class"].value_counts().sort_index()
print(counts5)
min_share = counts5.min() / counts5.sum()
if min_share < 0.005:
    raise RuntimeError(
        f"Class collapse: smallest class is {min_share:.2%} < 0.5%. "
        f"Re-run 01_v4_relabel_balanced.ipynb with adjusted thresholds.")

print("\ny_binary distribution:")
print(pdf["y_binary"].value_counts(normalize=True).round(4))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 33, Finished, Available, Finished, False)

y_5class distribution:
y_5class
0     2231
1     8156
2     7721
3    64375
4    17926
Name: count, dtype: int64

y_binary distribution:
y_binary
0    0.6692
1    0.3308
Name: proportion, dtype: float64


In [6]:
# ============================================================
# 3) FEATURE GROUPS — TRUE weekly columns + static
# ============================================================
def _by_suffix(suffix):
    return sorted([c for c in pdf.columns if c.endswith(suffix)])

WEEK_COLS = {
    1: _by_suffix("_week1"),
    2: _by_suffix("_week1") + _by_suffix("_week2"),
    3: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3"),
    4: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3") + _by_suffix("_week4"),
}

EXCLUDE = {
    "user_id", "course_id", "enroll_time",
    "is_completed", "completion_rate",
    "risk_label_5_name", "risk_label_5_idx",
    "label_D_flag", "label_E_flag",
    "y_binary", "y_5class",
    "total_4w_interactions", "post_w1_interactions", "perf_composite",
    "course_norm_score",
    "problems_passed", "course_raw_score_sum", "total_problems",
    "risk_label_5_surrogate",
}
ALL_WEEKLY = set().union(*WEEK_COLS.values())

STATIC_COLS = [
    c for c in pdf.columns
    if c not in EXCLUDE
    and c not in ALL_WEEKLY
    and pdf[c].dtype.kind in "biufc"
]

print(f"Static feature count: {len(STATIC_COLS)}")
print("Static cols:", STATIC_COLS[:20], "..." if len(STATIC_COLS) > 20 else "")
print()
for W, cols in WEEK_COLS.items():
    print(f"W={W}: {len(cols)} weekly columns up to and including week {W}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 34, Finished, Available, Finished, False)

Static feature count: 16
Static cols: ['year_of_birth', 'num_courses_enrolled', 'num_fields', 'num_resources', 'num_active_enrollments', 'course_total_videos', 'course_max_chapter', 'video_count_total_4w', 'video_watch_time_total_4w', 'chapter_coverage_ratio_raw', 'chapter_coverage_ratio', 'avg_chapter_progress_4w', 'chapter_progress_ratio_raw', 'chapter_progress_ratio', 'activity_signal_4w', 'is_active_4w'] 

W=1: 9 weekly columns up to and including week 1
W=2: 18 weekly columns up to and including week 2
W=3: 27 weekly columns up to and including week 3
W=4: 36 weekly columns up to and including week 4


In [7]:
# ============================================================
# 4) BUILD WINDOW VIEWS — leakage-safe (only weeks 1..W visible)
# ============================================================
def make_window_view(W):
    cols = STATIC_COLS + WEEK_COLS[W]
    cols = [c for c in cols if c in pdf.columns]
    Xdf  = pdf[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce").astype(float)
    return Xdf, cols

window_views = {}
for W in WINDOWS:
    Xdf, cols = make_window_view(W)
    window_views[W] = {"X": Xdf, "cols": cols}
    nan_pct = Xdf.isna().mean().mean()
    print(f"W={W}: shape={Xdf.shape}, mean NaN ratio={nan_pct:.3%}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 35, Finished, Available, Finished, False)

W=1: shape=(100409, 25), mean NaN ratio=27.326%
W=2: shape=(100409, 34), mean NaN ratio=43.076%
W=4: shape=(100409, 52), mean NaN ratio=59.910%


In [8]:
# ============================================================
# 5) STRATIFIED 8:1:1 SPLIT (per label)
# ============================================================
def split_idx(y, test_size=0.1, val_size=0.1, seed=SEED):
    idx = np.arange(len(y))
    idx_tv, idx_te = train_test_split(
        idx, test_size=test_size, stratify=y, random_state=seed)
    val_rel = val_size / (1.0 - test_size)
    idx_tr, idx_va = train_test_split(
        idx_tv, test_size=val_rel, stratify=y[idx_tv], random_state=seed)
    return idx_tr, idx_va, idx_te

splits = {}
for label in ["y_binary", "y_5class"]:
    y = pdf[label].values
    splits[label] = split_idx(y)
    n_tr, n_va, n_te = (len(splits[label][0]), len(splits[label][1]), len(splits[label][2]))
    print(f"{label}: train={n_tr}, val={n_va}, test={n_te}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 38, Finished, Available, Finished, False)

y_binary: train=80327, val=10041, test=10041
y_5class: train=80327, val=10041, test=10041


In [9]:
# ============================================================
# 6) METRIC HELPERS — EarlyRisk-MOOC metric set
# ============================================================
def evaluate_classifier(y_true, y_pred, y_proba=None, classes=None, name=""):
    res = {"name": name}
    res["accuracy"]    = accuracy_score(y_true, y_pred)
    res["macro_f1"]    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    res["weighted_f1"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if classes is None:
        classes = sorted(list(set(y_true) | set(y_pred)))
    per_class_f1 = f1_score(y_true, y_pred, average=None,
                            labels=classes, zero_division=0)
    for cls, f in zip(classes, per_class_f1):
        res[f"f1_{cls}"] = float(f)
    if y_proba is not None and len(classes) >= 2:
        try:
            if len(classes) == 2:
                res["auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                res["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr",
                    labels=classes, average="macro")
        except Exception as e:
            res["auc_error"] = str(e)
    return res


def flatten_result(res, name, classes_order, runtime, W, label_col):
    out = {"name": name, "label": label_col, "window": W,
           "runtime_sec": round(runtime, 2)}
    for k, v in res.items():
        if k.startswith("f1_") and k != "f1_score":
            try:
                idx = int(k.split("_")[1])
                out[f"f1_{classes_order[idx]}"] = v
            except (ValueError, IndexError):
                out[k] = v
        elif k in ("accuracy","macro_f1","weighted_f1","auc","auc_ovr"):
            out[k] = v
    return out


print("Metric helpers ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 44, Finished, Available, Finished, False)

Metric helpers ready


In [10]:
# ============================================================
# 7) IMPUTATION + SCALING (fit on TRAIN only)
# ============================================================
def fit_impute_scale(X_tr, X_te, X_va=None):
    imp = SimpleImputer(strategy="median").fit(X_tr)
    X_tr_i = imp.transform(X_tr)
    X_te_i = imp.transform(X_te)
    sc = StandardScaler().fit(X_tr_i)
    out = (sc.transform(X_tr_i), sc.transform(X_te_i))
    if X_va is not None:
        out = out + (sc.transform(imp.transform(X_va)),)
    return out

print("fit_impute_scale ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 50, Finished, Available, Finished, False)

fit_impute_scale ready


In [11]:
# ============================================================
# 8) XING & DU — Deep MLP + top-k intervention
# ============================================================
from sklearn.neural_network import MLPClassifier
TORCH_AVAILABLE = False
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    pass


if TORCH_AVAILABLE:
    class DeepMLP(nn.Module):
        def __init__(self, in_dim, n_cls):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32),      nn.ReLU(),
                nn.Linear(32, n_cls))
        def forward(self, x): return self.net(x)

    def train_dmlp(X, y, n_cls, epochs=20, lr=1e-3, batch=512):
        m = DeepMLP(X.shape[1], n_cls).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        crit = nn.CrossEntropyLoss()
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.long))
        dl = DataLoader(ds, batch_size=batch, shuffle=True)
        for _ in range(epochs):
            m.train()
            for a, b in dl:
                a, b = a.to(DEVICE), b.to(DEVICE)
                opt.zero_grad(); crit(m(a), b).backward(); opt.step()
        return m

    def pred_dmlp(m, X):
        m.eval()
        with torch.no_grad():
            p = torch.softmax(m(torch.tensor(X, dtype=torch.float32).to(DEVICE)), dim=1).cpu().numpy()
        return p.argmax(1), p


def topk_metrics(y_te, y_proba, classes_order):
    if classes_order == LABEL_ORDER_5:
        i_d, i_e = classes_order.index("D"), classes_order.index("E")
        r = y_proba[:, i_d] + y_proba[:, i_e]
        risk = ((y_te == i_d) | (y_te == i_e)).astype(int)
    else:
        r = y_proba[:, 0]
        risk = (y_te == 0).astype(int)
    out = {}
    n = len(r)
    for frac in [0.05, 0.10, 0.20]:
        k = max(1, int(n * frac))
        idx = np.argsort(-r)[:k]
        out[f"recall@{int(frac*100)}%"]    = float(risk[idx].sum() / max(1, risk.sum()))
        out[f"precision@{int(frac*100)}%"] = float(risk[idx].sum() / k)
    return out


all_results, prio_results = [], []
for label_col, classes in [("y_binary", [0,1]), ("y_5class", LABEL_ORDER_5)]:
    idx_tr, idx_va, idx_te = splits[label_col]
    y_all = pdf[label_col].values
    n_cls = len(classes)
    for W in WINDOWS:
        X = window_views[W]["X"].values
        X_tr_s, X_te_s = fit_impute_scale(X[idx_tr], X[idx_te])
        print(f"[{label_col}] W={W} DeepMLP ...")
        t0 = time.time()
        if TORCH_AVAILABLE:
            m = train_dmlp(X_tr_s, y_all[idx_tr], n_cls)
            y_pred, y_proba = pred_dmlp(m, X_te_s)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=80,
                                random_state=SEED, early_stopping=True).fit(X_tr_s, y_all[idx_tr])
            y_pred, y_proba = clf.predict(X_te_s), clf.predict_proba(X_te_s)
        rt = time.time()-t0
        res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                   classes=list(range(n_cls)), name="DeepMLP")
        row = flatten_result(res, "DeepMLP", classes, rt, W, label_col)
        all_results.append(row)
        prio = topk_metrics(y_all[idx_te], y_proba, classes)
        prio.update({"label": label_col, "window": W})
        prio_results.append(prio)
        print(f"   acc={row.get('accuracy',0):.4f}  macroF1={row.get('macro_f1',0):.4f}  recall@10%={prio['recall@10%']:.4f}")

res_df = pd.DataFrame(all_results)
prio_df = pd.DataFrame(prio_results)
print("\n=== Xing & Du (2019) — V4 classification ===")
print(res_df.round(4).to_string(index=False))
print("\n=== Top-k intervention metrics ===")
print(prio_df.round(4).to_string(index=False))
res_df.to_csv("/tmp/results_xing_du_2019.csv", index=False)
prio_df.to_csv("/tmp/results_xing_du_2019_priority.csv", index=False)
print("\nSaved -> /tmp/results_xing_du_2019*.csv")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 52, Finished, Available, Finished, False)

[y_binary] W=1 DeepMLP ...
   acc=0.8919  macroF1=0.8716  recall@10%=0.1490
[y_binary] W=2 DeepMLP ...
   acc=0.8971  macroF1=0.8788  recall@10%=0.1487
[y_binary] W=4 DeepMLP ...
   acc=0.9043  macroF1=0.8902  recall@10%=0.1490
[y_5class] W=1 DeepMLP ...
   acc=0.8000  macroF1=0.6520  recall@10%=0.1219
[y_5class] W=2 DeepMLP ...
   acc=0.8154  macroF1=0.6770  recall@10%=0.1219
[y_5class] W=4 DeepMLP ...
   acc=0.8520  macroF1=0.7420  recall@10%=0.1219

=== Xing & Du (2019) — V4 classification ===
   name    label  window  runtime_sec  accuracy  macro_f1  weighted_f1   f1_0   f1_1    auc   f1_A   f1_B   f1_C   f1_D   f1_E  auc_ovr
DeepMLP y_binary       1        41.47    0.8919    0.8716       0.8889 0.9227 0.8206 0.9485    NaN    NaN    NaN    NaN    NaN      NaN
DeepMLP y_binary       2        38.19    0.8971    0.8788       0.8947 0.9259 0.8316 0.9511    NaN    NaN    NaN    NaN    NaN      NaN
DeepMLP y_binary       4        34.73    0.9043    0.8902       0.9035 0.9295 0.8510 0.957